# DORA RAG — Phase 2: Chunking Strategy

This notebook covers:
- Loading cleaned documents from Phase 1
- Article-aware chunking (preserve legal structure)
- Sub-article chunking for long articles
- Metadata tagging for every chunk
- Chunk quality inspection
- Saving final chunks for Phase 3 (indexing)

**Why chunking matters for legal RAG:**
Generic sentence/token chunking destroys the legal context of DORA articles.
A chunk that cuts mid-paragraph or separates a paragraph from its Article header
will produce incomplete, potentially misleading answers.

## 2.0 Imports & Configuration

In [12]:
import re
import json
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Optional

# ── Paths (must match Phase 1 output) ────────────────────────────────────────
BASE_DIR    = Path("./dora_rag")
CLEANED_DIR = BASE_DIR / "dora_cleaned"
CHUNKS_DIR  = BASE_DIR / "dora_chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# ── Chunking parameters ───────────────────────────────────────────────────────
MAX_CHUNK_TOKENS   = 512    # target max tokens per chunk
OVERLAP_TOKENS     = 64     # overlap between sub-article chunks
CHARS_PER_TOKEN    = 4      # rough approximation for splitting
MAX_CHUNK_CHARS    = MAX_CHUNK_TOKENS * CHARS_PER_TOKEN   # ~2048
OVERLAP_CHARS      = OVERLAP_TOKENS  * CHARS_PER_TOKEN    # ~256

print(f"Chunking config:")
print(f"  Max chunk size : ~{MAX_CHUNK_TOKENS} tokens (~{MAX_CHUNK_CHARS} chars)")
print(f"  Overlap        : ~{OVERLAP_TOKENS} tokens (~{OVERLAP_CHARS} chars)")

Chunking config:
  Max chunk size : ~512 tokens (~2048 chars)
  Overlap        : ~64 tokens (~256 chars)


## 2.1 Load Cleaned Documents from Phase 1

In [13]:
# Load manifest
manifest_path = CLEANED_DIR / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Manifest not found at {manifest_path}.\n"
        "Please run 01_ingestion.ipynb first."
    )

manifest = json.loads(manifest_path.read_text())

# Load each document's cleaned text and article map
docs = {}
for entry in manifest:
    name = entry["name"]
    txt_path = Path(entry["txt_file"])
    
    if not txt_path.exists():
        print(f"  ✗ Skipping {name} — file not found: {txt_path}")
        continue
    
    cleaned_text = txt_path.read_text(encoding="utf-8")
    
    article_map = []
    if entry.get("article_map_file"):
        map_path = Path(entry["article_map_file"])
        if map_path.exists():
            article_map = json.loads(map_path.read_text())
    
    docs[name] = {
        **entry,
        "cleaned_text": cleaned_text,
        "article_map": article_map
    }
    print(f"  ✓ Loaded: {name} ({len(cleaned_text):,} chars, {len(article_map)} articles)")

print(f"\nLoaded {len(docs)} documents")

  ✓ Loaded: L1_DORA_Regulation_2022_2554 (304,419 chars, 66 articles)
  ✓ Loaded: L1_DORA_Directive_2022_2556 (37,960 chars, 12 articles)
  ✓ Loaded: L2_RTS_ICT_Risk_Management (123,338 chars, 42 articles)
  ✓ Loaded: L2_RTS_ICT_Incident_Classification (31,277 chars, 13 articles)
  ✓ Loaded: L2_RTS_ICT_Incident_Reporting_Process (16,759 chars, 7 articles)
  ✓ Loaded: L2_ITS_ICT_Incident_Reporting (96,582 chars, 9 articles)
  ✓ Loaded: L2_RTS_TLPT (30,899 chars, 0 articles)
  ✓ Loaded: L2_RTS_ICT_Third_Party_Policy (29,979 chars, 4 articles)
  ✓ Loaded: L2_RTS_Subcontracting (25,097 chars, 7 articles)
  ✓ Loaded: L2_ITS_Register_of_Information (103,606 chars, 8 articles)
  ✓ Loaded: L2_DR_CTPP_Designation_Criteria (31,779 chars, 7 articles)
  ✓ Loaded: L2_DR_DORA_Oversight_Fees (21,823 chars, 7 articles)
  ✓ Loaded: L2_RTS_Oversight_Conditions (31,139 chars, 7 articles)
  ✓ Loaded: L2_RTS_Joint_Examination_Team (21,104 chars, 6 articles)
  ✓ Loaded: L3_Guidelines_Costs_and_Losses (11,04

## 2.2 Define the Chunk Data Structure

A `DoraChunk` holds the text content and all metadata needed for retrieval,
filtering, and citation generation.

In [14]:
@dataclass
class DoraChunk:
    """
    A single retrievable chunk from the DORA corpus.
    
    Fields
    ------
    chunk_id        : unique identifier (e.g. 'DORA_2022_2554_art28_p0')
    text            : the chunk content (used for embedding + generation)
    text_with_header: text prefixed with Article header (used in prompts)
    source_doc      : name of source document
    doc_type        : 'regulation' | 'RTS' | 'ITS' | 'guidance'
    article_num     : integer article number (None for recitals/annexes)
    article_title   : title of the article
    paragraph_idx   : which sub-chunk within this article (0-indexed)
    chapter         : chapter reference (if extractable)
    topic_tags      : list of topic labels
    entity_scope    : list of entity types this applies to
    char_start      : character offset in source text
    char_end        : character offset in source text
    token_estimate  : rough token count
    """
    chunk_id:         str
    text:             str
    text_with_header: str
    source_doc:       str
    doc_type:         str
    article_num:      Optional[int]
    article_title:    str
    paragraph_idx:    int
    chapter:          Optional[str]
    topic_tags:       List[str]
    entity_scope:     List[str]
    char_start:       int
    char_end:         int
    token_estimate:   int


def estimate_tokens(text: str) -> int:
    """Rough token estimate: chars / 4."""
    return len(text) // 4


print("DoraChunk dataclass defined.")
print("Fields:", [f.name for f in DoraChunk.__dataclass_fields__.values()])

DoraChunk dataclass defined.
Fields: ['chunk_id', 'text', 'text_with_header', 'source_doc', 'doc_type', 'article_num', 'article_title', 'paragraph_idx', 'chapter', 'topic_tags', 'entity_scope', 'char_start', 'char_end', 'token_estimate']


## 2.3 Topic & Entity Tagging

Lightweight keyword-based tagging — no ML needed at baseline.
This enables metadata filtering during retrieval (e.g. 'only show
chunks about third-party risk for investment firms').

In [15]:
# ── Topic keyword map ─────────────────────────────────────────────────────────
TOPIC_KEYWORDS = {
    "ict_risk_management":    ["ict risk", "information and communication", "risk management framework",
                                "risk appetite", "risk tolerance"],
    "incident_reporting":     ["incident", "major ict", "reporting", "notification", "classify"],
    "third_party_risk":       ["third-party", "third party", "ict third", "outsourc", "sub-outsourc",
                                "service provider", "ctpp", "critical third"],
    "resilience_testing":     ["resilience testing", "tlpt", "penetration test", "vulnerability",
                                "tiber", "threat-led"],
    "governance":             ["management body", "board", "governance", "oversight", "accountability",
                                "responsibility"],
    "business_continuity":    ["business continuity", "disaster recovery", "backup", "crisis",
                                "continuity plan"],
    "information_sharing":    ["information sharing", "threat intelligence", "cyber threat", "share"],
    "data_security":          ["data security", "encryption", "access control", "authentication",
                                "data integrity", "confidentiality"],
    "supervision":            ["competent authorit", "supervisor", "eba", "eiopa", "esma",
                                "oversight framework", "lead overseer"],
}

# ── Entity scope keyword map ──────────────────────────────────────────────────
ENTITY_KEYWORDS = {
    "credit_institutions":    ["credit institution", "bank"],
    "investment_firms":       ["investment firm"],
    "insurance":              ["insurance", "reinsurance", "insurer"],
    "payment_institutions":   ["payment institution", "e-money", "electronic money"],
    "crypto_assets":          ["crypto-asset", "crypto asset", "casp"],
    "ict_third_party":        ["ict third-party provider", "ctpp", "critical third-party"],
    "simplified_framework":   ["simplified ict risk", "small and non-interconnected"],
}


def infer_topic_tags(text: str) -> List[str]:
    text_lower = text.lower()
    return [
        topic for topic, keywords in TOPIC_KEYWORDS.items()
        if any(kw in text_lower for kw in keywords)
    ]


def infer_entity_scope(text: str) -> List[str]:
    text_lower = text.lower()
    return [
        entity for entity, keywords in ENTITY_KEYWORDS.items()
        if any(kw in text_lower for kw in keywords)
    ]


# Quick test
test_text = """Financial entities shall report major ICT-related incidents to the 
competent authority within 4 hours of classification."""
print("Test text:", test_text[:80])
print("Topics:  ", infer_topic_tags(test_text))
print("Entities:", infer_entity_scope(test_text))

Test text: Financial entities shall report major ICT-related incidents to the 
competent au
Topics:   ['incident_reporting', 'supervision']
Entities: []


## 2.4 Core Chunking Function

Strategy:
1. **Short articles** (≤ MAX_CHUNK_CHARS) → single chunk, whole article
2. **Long articles** → split on paragraph boundaries with overlap
3. Each chunk includes Article header as a prefix for context

In [16]:
def split_into_paragraphs(text: str) -> List[str]:
    """Split article text into paragraphs on double newlines."""
    paragraphs = re.split(r'\n{2,}', text.strip())
    # Filter out very short fragments (likely artefacts)
    return [p.strip() for p in paragraphs if len(p.strip()) > 40]


def make_chunk_id(doc_name: str, article_num: Optional[int],
                  para_idx: int, doc_type: str) -> str:
    doc_short = doc_name[:20].replace(" ", "_")
    art_part  = f"art{article_num:03d}" if article_num else "preamble"
    return f"{doc_short}__{art_part}__p{para_idx:02d}"


def chunk_article(
    article_text: str,
    article_num: int,
    article_title: str,
    source_doc: str,
    doc_type: str,
    char_start: int,
) -> List[DoraChunk]:
    """
    Chunk a single article into one or more DoraChunks.
    Short articles → 1 chunk.
    Long articles → paragraph-boundary chunks with overlap.
    """
    header = f"Article {article_num} — {article_title}\n\n"
    chunks = []

    # Case 1: Article fits in a single chunk
    if len(article_text) <= MAX_CHUNK_CHARS:
        chunk_text = article_text
        chunks.append(DoraChunk(
            chunk_id         = make_chunk_id(source_doc, article_num, 0, doc_type),
            text             = chunk_text,
            text_with_header = header + chunk_text,
            source_doc       = source_doc,
            doc_type         = doc_type,
            article_num      = article_num,
            article_title    = article_title,
            paragraph_idx    = 0,
            chapter          = None,   # enriched later if needed
            topic_tags       = infer_topic_tags(chunk_text),
            entity_scope     = infer_entity_scope(chunk_text),
            char_start       = char_start,
            char_end         = char_start + len(article_text),
            token_estimate   = estimate_tokens(chunk_text),
        ))
        return chunks

    # Case 2: Article is long — split on paragraphs with overlap
    paragraphs = split_into_paragraphs(article_text)
    current_chunk_paras = []
    current_chars = 0
    chunk_idx = 0

    for para in paragraphs:
        para_chars = len(para)

        # If adding this paragraph exceeds limit, flush current chunk
        if current_chars + para_chars > MAX_CHUNK_CHARS and current_chunk_paras:
            chunk_text = "\n\n".join(current_chunk_paras)
            chunks.append(DoraChunk(
                chunk_id         = make_chunk_id(source_doc, article_num, chunk_idx, doc_type),
                text             = chunk_text,
                text_with_header = header + chunk_text,
                source_doc       = source_doc,
                doc_type         = doc_type,
                article_num      = article_num,
                article_title    = article_title,
                paragraph_idx    = chunk_idx,
                chapter          = None,
                topic_tags       = infer_topic_tags(chunk_text),
                entity_scope     = infer_entity_scope(chunk_text),
                char_start       = char_start,
                char_end         = char_start + len(chunk_text),
                token_estimate   = estimate_tokens(chunk_text),
            ))
            chunk_idx += 1

            # Overlap: keep last paragraph(s) into next chunk
            overlap_paras = []
            overlap_chars = 0
            for p in reversed(current_chunk_paras):
                if overlap_chars + len(p) <= OVERLAP_CHARS:
                    overlap_paras.insert(0, p)
                    overlap_chars += len(p)
                else:
                    break
            current_chunk_paras = overlap_paras
            current_chars = overlap_chars

        current_chunk_paras.append(para)
        current_chars += para_chars

    # Flush remaining paragraphs
    if current_chunk_paras:
        chunk_text = "\n\n".join(current_chunk_paras)
        chunks.append(DoraChunk(
            chunk_id         = make_chunk_id(source_doc, article_num, chunk_idx, doc_type),
            text             = chunk_text,
            text_with_header = header + chunk_text,
            source_doc       = source_doc,
            doc_type         = doc_type,
            article_num      = article_num,
            article_title    = article_title,
            paragraph_idx    = chunk_idx,
            chapter          = None,
            topic_tags       = infer_topic_tags(chunk_text),
            entity_scope     = infer_entity_scope(chunk_text),
            char_start       = char_start,
            char_end         = char_start + len(chunk_text),
            token_estimate   = estimate_tokens(chunk_text),
        ))

    return chunks


print("Chunking functions defined.")

Chunking functions defined.


## 2.5 Run Chunking on All Documents

In [17]:
all_chunks: List[DoraChunk] = []

for doc_name, doc in docs.items():
    text        = doc["cleaned_text"]
    article_map = doc["article_map"]
    doc_type    = doc["doc_type"]
    doc_chunks  = []

    if not article_map:
        # No article map — fall back to sliding window chunking
        print(f"  {doc_name}: no article map, using sliding window")
        start = 0
        idx   = 0
        while start < len(text):
            end        = min(start + MAX_CHUNK_CHARS, len(text))
            chunk_text = text[start:end]
            doc_chunks.append(DoraChunk(
                chunk_id         = make_chunk_id(doc_name, None, idx, doc_type),
                text             = chunk_text,
                text_with_header = chunk_text,
                source_doc       = doc_name,
                doc_type         = doc_type,
                article_num      = None,
                article_title    = "Unknown",
                paragraph_idx    = idx,
                chapter          = None,
                topic_tags       = infer_topic_tags(chunk_text),
                entity_scope     = infer_entity_scope(chunk_text),
                char_start       = start,
                char_end         = end,
                token_estimate   = estimate_tokens(chunk_text),
            ))
            start += MAX_CHUNK_CHARS - OVERLAP_CHARS
            idx   += 1
    else:
        # Article-aware chunking
        for article in article_map:
            art_text  = text[article["char_start"]:article["char_end"]]
            art_chunks = chunk_article(
                article_text  = art_text,
                article_num   = article["article_num"],
                article_title = article["title"],
                source_doc    = doc_name,
                doc_type      = doc_type,
                char_start    = article["char_start"],
            )
            doc_chunks.extend(art_chunks)

    all_chunks.extend(doc_chunks)
    avg_tokens = sum(c.token_estimate for c in doc_chunks) / len(doc_chunks) if doc_chunks else 0
    print(f"  {doc_name}: {len(doc_chunks)} chunks (avg ~{avg_tokens:.0f} tokens)")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

  L1_DORA_Regulation_2022_2554: 116 chunks (avg ~582 tokens)
  L1_DORA_Directive_2022_2556: 16 chunks (avg ~379 tokens)
  L2_RTS_ICT_Risk_Management: 63 chunks (avg ~388 tokens)
  L2_RTS_ICT_Incident_Classification: 14 chunks (avg ~308 tokens)
  L2_RTS_ICT_Incident_Reporting_Process: 8 chunks (avg ~355 tokens)
  L2_ITS_ICT_Incident_Reporting: 47 chunks (avg ~483 tokens)
  L2_RTS_TLPT: no article map, using sliding window
  L2_RTS_TLPT: 18 chunks (avg ~490 tokens)
  L2_RTS_ICT_Third_Party_Policy: 8 chunks (avg ~497 tokens)
  L2_RTS_Subcontracting: 10 chunks (avg ~416 tokens)
  L2_ITS_Register_of_Information: 47 chunks (avg ~486 tokens)
  L2_DR_CTPP_Designation_Criteria: 7 chunks (avg ~459 tokens)
  L2_DR_DORA_Oversight_Fees: 7 chunks (avg ~286 tokens)
  L2_RTS_Oversight_Conditions: 12 chunks (avg ~462 tokens)
  L2_RTS_Joint_Examination_Team: 7 chunks (avg ~417 tokens)
  L3_Guidelines_Costs_and_Losses: no article map, using sliding window
  L3_Guidelines_Costs_and_Losses: 7 chunks (avg ~

## 2.6 Chunk Quality Analysis

Before indexing, inspect the distribution of chunk sizes and
verify metadata is populated correctly.

In [18]:
import statistics

token_sizes = [c.token_estimate for c in all_chunks]

print("=" * 55)
print("CHUNK SIZE DISTRIBUTION (token estimates)")
print("=" * 55)
print(f"Total chunks  : {len(token_sizes)}")
print(f"Min tokens    : {min(token_sizes)}")
print(f"Max tokens    : {max(token_sizes)}")
print(f"Mean tokens   : {statistics.mean(token_sizes):.0f}")
print(f"Median tokens : {statistics.median(token_sizes):.0f}")
print(f"Stdev tokens  : {statistics.stdev(token_sizes):.0f}")

# Bucket distribution
buckets = [(0,100), (100,200), (200,300), (300,400), (400,512), (512,99999)]
print("\nBucket distribution:")
for lo, hi in buckets:
    count = sum(1 for t in token_sizes if lo <= t < hi)
    bar = '█' * (count * 40 // max(1, len(token_sizes)))
    label = f"{lo:>4}–{hi if hi < 9999 else '∞':>4} tokens"
    print(f"  {label}: {bar} {count}")

# Flag oversized chunks
oversized = [c for c in all_chunks if c.token_estimate > MAX_CHUNK_TOKENS]
print(f"\nOversized chunks (>{MAX_CHUNK_TOKENS} tokens): {len(oversized)}")
if oversized:
    for c in oversized[:3]:
        print(f"  {c.chunk_id} — {c.token_estimate} tokens")

CHUNK SIZE DISTRIBUTION (token estimates)
Total chunks  : 390
Min tokens    : 11
Max tokens    : 1432
Mean tokens   : 476
Median tokens : 477
Stdev tokens  : 280

Bucket distribution:
     0– 100 tokens: ██ 26
   100– 200 tokens: ████ 41
   200– 300 tokens: █████ 54
   300– 400 tokens: ████ 40
   400– 512 tokens: █████ 58
   512–   ∞ tokens: █████████████████ 171

Oversized chunks (>512 tokens): 145
  L1_DORA_Regulation_2__art006__p01 — 1326 tokens
  L1_DORA_Regulation_2__art006__p02 — 1315 tokens
  L1_DORA_Regulation_2__art006__p03 — 1141 tokens


In [19]:
# ── Metadata coverage ─────────────────────────────────────────────────────────
print("METADATA COVERAGE")
print("-" * 40)

with_articles   = sum(1 for c in all_chunks if c.article_num is not None)
with_topics     = sum(1 for c in all_chunks if c.topic_tags)
with_entities   = sum(1 for c in all_chunks if c.entity_scope)

print(f"Chunks with article_num : {with_articles}/{len(all_chunks)} "
      f"({100*with_articles//len(all_chunks)}%)")
print(f"Chunks with topic_tags  : {with_topics}/{len(all_chunks)} "
      f"({100*with_topics//len(all_chunks)}%)")
print(f"Chunks with entity_scope: {with_entities}/{len(all_chunks)} "
      f"({100*with_entities//len(all_chunks)}%)")

# Topic distribution
from collections import Counter
all_topics = [t for c in all_chunks for t in c.topic_tags]
topic_counts = Counter(all_topics).most_common()

print("\nTopic tag distribution:")
for topic, count in topic_counts:
    print(f"  {topic:<30}: {count} chunks")

METADATA COVERAGE
----------------------------------------
Chunks with article_num : 362/390 (92%)
Chunks with topic_tags  : 348/390 (89%)
Chunks with entity_scope: 84/390 (21%)

Topic tag distribution:
  third_party_risk              : 192 chunks
  incident_reporting            : 177 chunks
  supervision                   : 142 chunks
  ict_risk_management           : 87 chunks
  governance                    : 82 chunks
  data_security                 : 55 chunks
  information_sharing           : 53 chunks
  business_continuity           : 47 chunks
  resilience_testing            : 28 chunks


## 2.7 Inspect Sample Chunks

Manually review a few chunks from key articles to confirm quality.

In [20]:
def show_chunk(chunk: DoraChunk):
    print(f"{'─'*70}")
    print(f"ID           : {chunk.chunk_id}")
    print(f"Article      : {chunk.article_num} — {chunk.article_title}")
    print(f"Doc          : {chunk.source_doc} ({chunk.doc_type})")
    print(f"Tokens (est) : {chunk.token_estimate}")
    print(f"Topics       : {chunk.topic_tags}")
    print(f"Entities     : {chunk.entity_scope}")
    print(f"\nText preview (first 600 chars):")
    print(chunk.text_with_header[:600])
    print()


# Show chunks from key articles: 17 (incidents), 28 (third-party), 24 (testing)
key_articles = [17, 24, 28]
shown = set()

for art_num in key_articles:
    for chunk in all_chunks:
        if chunk.article_num == art_num and art_num not in shown:
            show_chunk(chunk)
            shown.add(art_num)
            break

if not shown:
    # Fall back: show first 3 chunks
    print("Key articles not found — showing first 3 chunks")
    for chunk in all_chunks[:3]:
        show_chunk(chunk)

──────────────────────────────────────────────────────────────────────
ID           : L1_DORA_Regulation_2__art017__p00
Article      : 17 — ICT-related incident management process
Doc          : L1_DORA_Regulation_2022_2554 (regulation)
Tokens (est) : 466
Topics       : ['incident_reporting', 'governance', 'information_sharing']
Entities     : []

Text preview (first 600 chars):
Article 17 — ICT-related incident management process

Article 17
ICT-related incident management process
1. Financial entities shall define, establish and implement an ICT-related incident management process to detect, manage
and notify ICT-related incidents.
2. Financial entities shall record all ICT-related incidents and significant cyber threats. Financial entities shall establish
appropriate procedures and processes to ensure a consistent and integrated monitoring, handling and follow-up of ICTrelated incidents, to ensure that root causes are identified, documented and addressed in order t

────────────────

## 2.8 Save Chunks to Disk

Persist as JSON for Phase 3 (embedding + indexing).

In [21]:
# ── Serialise all chunks ──────────────────────────────────────────────────────
chunks_data = [asdict(c) for c in all_chunks]

chunks_path = CHUNKS_DIR / "dora_chunks.json"
chunks_path.write_text(json.dumps(chunks_data, indent=2, ensure_ascii=False))

file_size_mb = chunks_path.stat().st_size / (1024 * 1024)
print(f"✓ Saved {len(chunks_data)} chunks to: {chunks_path}")
print(f"  File size: {file_size_mb:.2f} MB")

# ── Also save a lightweight index (chunk_id + metadata only, no text) ─────────
index_data = [
    {
        "chunk_id":      c["chunk_id"],
        "source_doc":    c["source_doc"],
        "doc_type":      c["doc_type"],
        "article_num":   c["article_num"],
        "article_title": c["article_title"],
        "paragraph_idx": c["paragraph_idx"],
        "topic_tags":    c["topic_tags"],
        "entity_scope":  c["entity_scope"],
        "token_estimate":c["token_estimate"],
    }
    for c in chunks_data
]
index_path = CHUNKS_DIR / "dora_chunks_index.json"
index_path.write_text(json.dumps(index_data, indent=2))
print(f"✓ Saved metadata index to: {index_path}")

✓ Saved 390 chunks to: dora_rag/dora_chunks/dora_chunks.json
  File size: 1.65 MB
✓ Saved metadata index to: dora_rag/dora_chunks/dora_chunks_index.json


## 2.9 Phase 2 Summary

In [22]:
print("\n" + "=" * 60)
print("PHASE 2 COMPLETE — CHUNKING SUMMARY")
print("=" * 60)

by_doc = Counter(c.source_doc for c in all_chunks)
print(f"\n{'Document':<40} {'Chunks':>6}")
print("-" * 48)
for doc_name, count in by_doc.most_common():
    print(f"{doc_name[:40]:<40} {count:>6}")
print("-" * 48)
print(f"{'TOTAL':<40} {len(all_chunks):>6}")

print(f"""
Chunk config used:
  Strategy     : Article-aware + paragraph splitting
  Max tokens   : ~{MAX_CHUNK_TOKENS}
  Overlap      : ~{OVERLAP_TOKENS} tokens
  Metadata     : article_num, title, topic_tags, entity_scope

Output files:
  {chunks_path}
  {index_path}

Next step → open 03_indexing.ipynb
""")


PHASE 2 COMPLETE — CHUNKING SUMMARY

Document                                 Chunks
------------------------------------------------
L1_DORA_Regulation_2022_2554                116
L2_RTS_ICT_Risk_Management                   63
L2_ITS_ICT_Incident_Reporting                47
L2_ITS_Register_of_Information               47
L2_RTS_TLPT                                  18
L1_DORA_Directive_2022_2556                  16
L2_RTS_ICT_Incident_Classification           14
L2_RTS_Oversight_Conditions                  12
L2_RTS_Subcontracting                        10
L2_RTS_ICT_Incident_Reporting_Process         8
L2_RTS_ICT_Third_Party_Policy                 8
L2_DR_CTPP_Designation_Criteria               7
L2_DR_DORA_Oversight_Fees                     7
L2_RTS_Joint_Examination_Team                 7
L3_Guidelines_Costs_and_Losses                7
L3_ESAs_Public_Statement_DORA                 3
------------------------------------------------
TOTAL                                       390
